# parameters_and_arguments
**Prerequisites:** function_basics

**Outcome:** After this notebook you will know the exact difference between a parameter and an argument, every way Python lets you pass data into a function, and how to design function signatures that are clear, flexible, and hard to misuse.

## The Problem

In [ ]:
# what is wrong with each of these calls?
def create_job(job_id, status, region, retries):
    return {"id": job_id, "status": status, "region": region, "retries": retries}

# call 1 — wrong order, no error, wrong data silently loaded
job = create_job("pending", "job_1", 3, "us-east")
print(job)

# call 2 — too many arguments
try:
    job = create_job("job_1", "pending", "us-east", 3, "extra")
except TypeError as e:
    print(e)

# call 3 — missing argument
try:
    job = create_job("job_1", "pending")
except TypeError as e:
    print(e)

## The Concept

A parameter is the name listed in the function definition. An argument is the value you pass when you call the function. Python matches arguments to parameters either by position — the first argument goes to the first parameter — or by keyword — you name the parameter explicitly in the call. Beyond positional and keyword arguments, Python provides default values, `*args` for variable positional arguments, `**kwargs` for variable keyword arguments, and positional-only and keyword-only markers. Together these give you complete control over how your functions are called.

## Minimal Example

In [ ]:
def create_job(job_id, status, region):
    return {"id": job_id, "status": status, "region": region}

# positional — order matters
print(create_job("job_1", "pending", "us-east"))

# keyword — order does not matter
print(create_job(region="eu-west", status="done", job_id="job_2"))

# mixed — positional first, keyword after
print(create_job("job_3", region="us-east", status="failed"))

## How Python Does It

When Python calls a function it creates a new local namespace and binds each parameter name to the corresponding argument value. Positional arguments are matched left to right by index. Keyword arguments are matched by name and can appear in any order. If a parameter has a default value and no argument is supplied for it, the default is used. If a required parameter receives no argument Python raises `TypeError` before the function body executes.

In [ ]:
import inspect

def create_job(job_id, status, region="us-east", retries=3):
    return {"id": job_id, "status": status, "region": region, "retries": retries}

sig = inspect.signature(create_job)
for name, param in sig.parameters.items():
    print(f"{name:10} default={param.default!r:12} kind={param.kind.name}")

## Building Up

In [ ]:
# default arguments — make parameters optional
def connect(host, port=5432, timeout=30):
    return f"connecting to {host}:{port} timeout={timeout}s"

print(connect("localhost"))                   # both defaults used
print(connect("localhost", 9090))             # port overridden
print(connect("localhost", timeout=10))       # timeout overridden by keyword

In [ ]:
# *args — collect any number of positional arguments into a tuple
def log_events(*events):
    for event in events:
        print(f"[EVENT] {event}")

log_events("boot")
log_events("boot", "connected", "synced")
log_events()   # zero arguments — events is an empty tuple

In [ ]:
# **kwargs — collect any number of keyword arguments into a dict
def create_record(**fields):
    return fields

print(create_record(id="job_1", status="pending", region="us-east"))
print(create_record(id="job_2", status="done"))

In [ ]:
# combining all parameter types — fixed order required
# positional, *args, keyword-with-default, **kwargs
def pipeline(name, *stages, region="us-east", **options):
    return {
        "name":    name,
        "stages":  stages,
        "region":  region,
        "options": options
    }

result = pipeline(
    "etl",
    "extract", "transform", "load",
    region="eu-west",
    retries=3,
    dry_run=True
)
print(result)

In [ ]:
# keyword-only parameters — enforced by placing after *
def run_job(job_id, *, region, dry_run=False):
    return {"id": job_id, "region": region, "dry_run": dry_run}

print(run_job("job_1", region="us-east"))
print(run_job("job_2", region="eu-west", dry_run=True))

try:
    run_job("job_3", "us-east")   # region cannot be passed positionally
except TypeError as e:
    print(e)

In [ ]:
# positional-only parameters — enforced by placing before /
def connect(host, port, /, timeout=30):
    return f"{host}:{port} t={timeout}"

print(connect("localhost", 5432))              # positional — fine
print(connect("localhost", 5432, timeout=10))  # timeout by keyword — fine

try:
    connect(host="localhost", port=5432)        # host and port are positional-only
except TypeError as e:
    print(e)

In [ ]:
# unpacking a list or dict into arguments at the call site
def create_job(job_id, status, region):
    return {"id": job_id, "status": status, "region": region}

args   = ["job_1", "pending"]
kwargs = {"region": "us-east"}

print(create_job(*args, **kwargs))   # unpacked at call time

## Where It Breaks

In [ ]:
# positional argument after keyword argument
def create_job(job_id, status, region):
    return {"id": job_id, "status": status, "region": region}

try:
    create_job(job_id="job_1", "pending", "us-east")   # SyntaxError
except SyntaxError as e:
    print(e)

In [ ]:
# same argument passed twice
def create_job(job_id, status, region):
    return {"id": job_id, "status": status, "region": region}

try:
    create_job("job_1", "pending", "us-east", job_id="job_2")
except TypeError as e:
    print(e)

In [ ]:
# default parameter placed before non-default — SyntaxError
try:
    exec("def bad(a=1, b): pass")
except SyntaxError as e:
    print(e)

## The Fix

In [ ]:
# use keyword arguments for functions with many parameters
# makes call sites self-documenting and order-independent
def create_job(job_id, status, region="us-east", retries=3):
    return {"id": job_id, "status": status, "region": region, "retries": retries}

# hard to misread — every argument is named
job = create_job(
    job_id="job_1",
    status="pending",
    region="eu-west",
    retries=5
)
print(job)

## In a Real System

In [ ]:
# a flexible pipeline runner that accepts a variable number of stages
# and arbitrary configuration options without changing its signature
def run_pipeline(name, *stages, dry_run=False, **config):
    print(f"pipeline : {name}")
    print(f"stages   : {stages}")
    print(f"dry_run  : {dry_run}")
    print(f"config   : {config}")
    if dry_run:
        print("dry run — no changes written")
        return
    for stage in stages:
        print(f"  running stage: {stage}")

run_pipeline(
    "etl",
    "extract", "transform", "load",
    dry_run=True,
    region="us-east",
    retries=3
)

## Performance

Positional arguments are slightly faster than keyword arguments because Python can match them by index without a dictionary lookup. The difference is nanoseconds and irrelevant in almost all code. `*args` builds a tuple — O(n) where n is the number of extra arguments. `**kwargs` builds a dict — O(n) as well. In extremely hot loops with known argument counts, using positional arguments avoids both allocations entirely.

## Anti-Pattern

In [ ]:
# anti-pattern: using **kwargs to hide a function's real interface
def run_job(**kwargs):          # caller has no idea what is required
    job_id  = kwargs["job_id"]
    region  = kwargs["region"]
    retries = kwargs.get("retries", 3)
    return {"id": job_id, "region": region, "retries": retries}

# correct: explicit parameters make the interface visible
def run_job(job_id, region, retries=3):
    return {"id": job_id, "region": region, "retries": retries}

# use **kwargs only when the set of keys is genuinely unknown at definition time
print(run_job("job_1", "us-east"))

## Interview Signals

- What is the difference between a parameter and an argument?
- What is the difference between `*args` and `**kwargs` and when would you use each?
- What does the `/` marker in a function signature enforce?
- What does the `*` marker in a function signature enforce?

## Exercise

In [ ]:
def build_endpoint(host, port, *paths, secure=False, **params):
    """
    Builds a URL string from the given components.

    - scheme is 'https' if secure=True else 'http'
    - paths are joined with '/'
    - params are appended as query string key=value pairs joined with '&'
    - return empty string if host is empty

    Example:
        build_endpoint('api.example.com', 443, 'v1', 'jobs',
                       secure=True, region='us-east', limit='10')
        => 'https://api.example.com:443/v1/jobs?region=us-east&limit=10'

    Bug: the function body is missing — implement it.
    """
    pass


assert build_endpoint(
    "api.example.com", 443, "v1", "jobs",
    secure=True, region="us-east", limit="10"
) == "https://api.example.com:443/v1/jobs?region=us-east&limit=10"

assert build_endpoint(
    "localhost", 8080, "health"
) == "http://localhost:8080/health"

assert build_endpoint(
    "localhost", 8080
) == "http://localhost:8080"

assert build_endpoint("", 80) == ""

print("all assertions passed")